# PlantMetWiki figures

Generates publication-quality figures from **live SPARQL queries** against the Virtuoso endpoint.

**Workflow:**
```
Virtuoso endpoint  →  SPARQL queries  →  fresh CSVs (figures/output/)  →  figures (figures/output/)
```

Reference CSVs from a previous Snorql run are in `figures/reference/` for comparison only — they are **never used as inputs here**.

**Kernel:** `plantmetwiki-rdf`  
**Set `SPARQL_ENDPOINT` below before running.**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from SPARQLWrapper import SPARQLWrapper, CSV
import io

# ── Configure ─────────────────────────────────────────────────────────────────
SPARQL_ENDPOINT = "https://sparql-plantmetwiki.bioinformatics.nl/sparql"

OUT_DIR = Path("figures/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

# Named graphs
G_CORE     = "http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways"
G_TAXONOMY = "http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra"

def run_query(query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT and return a DataFrame. Raises on error."""
    sw = SPARQLWrapper(SPARQL_ENDPOINT)
    sw.setReturnFormat(CSV)
    sw.setQuery(query)
    result = sw.query().convert()
    return pd.read_csv(io.BytesIO(result))

def save_csv(df: pd.DataFrame, name: str) -> pd.DataFrame:
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print(f"  Saved {len(df):,} rows → {path}")
    return df

def save_fig(fig: plt.Figure, name: str) -> None:
    for ext in ("pdf", "svg"):
        fig.savefig(OUT_DIR / f"{name}.{ext}", bbox_inches="tight")
    print(f"  Saved → {OUT_DIR}/{name}.{{pdf,svg}}")

def find_col(df: pd.DataFrame, needle: str) -> str:
    needle = needle.lower()
    for c in df.columns:
        if needle in c.lower():
            return c
    raise ValueError(f"No column containing '{needle}'. Columns: {list(df.columns)}")

print(f"Endpoint : {SPARQL_ENDPOINT}")
print(f"Output   : {OUT_DIR.resolve()}")

---
## 1. Fetch fresh data from Virtuoso

Each cell queries the live endpoint and saves a CSV to `figures/output/`.

In [ ]:
print("Querying genes per pathway...")
genes = save_csv(run_query(f"""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {{
  GRAPH <{G_CORE}> {{
    ?node a wp:GeneProduct ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }}
}}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "genes_per_pathway.csv")
genes.head(3)

In [ ]:
print("Querying metabolites per pathway...")
metabolites = save_csv(run_query(f"""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {{
  GRAPH <{G_CORE}> {{
    ?node a wp:Metabolite ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }}
}}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "metabolites_per_pathway.csv")
metabolites.head(3)

In [ ]:
print("Querying enzymes (proteins) per pathway...")
enzymes = save_csv(run_query(f"""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {{
  GRAPH <{G_CORE}> {{
    ?node a wp:Protein ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }}
}}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "enzymes_per_pathway.csv")
enzymes.head(3)

In [ ]:
print("Querying conversions per pathway...")
conversions = save_csv(run_query(f"""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?interaction) AS ?count)
WHERE {{
  GRAPH <{G_CORE}> {{
    ?interaction a wp:Conversion ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }}
}}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "conversions_per_pathway.csv")
conversions.head(3)

In [ ]:
print("Querying distinct species per pathway...")
species_pw = save_csv(run_query(f"""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi: <http://purl.obolibrary.org/obo/NCBITaxon_>

SELECT ?pwID (COUNT(DISTINCT ?species) AS ?count)
WHERE {{
  GRAPH <{G_TAXONOMY}> {{
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
  GRAPH <{G_CORE}> {{
    ?node wp:isPartOf ?pwID .
  }}
}}
GROUP BY ?pwID
ORDER BY DESC(?count)
"""), "species_per_pathway.csv")
species_pw.head(3)

In [ ]:
print("Querying interaction type counts...")
int_types = save_csv(run_query(f"""
PREFIX wp:  <http://vocabularies.wikipathways.org/wp#>

SELECT ?type (COUNT(?i) AS ?n)
WHERE {{
  GRAPH <{G_CORE}> {{
    ?i a ?type .
    FILTER(STRSTARTS(STR(?type), "http://vocabularies.wikipathways.org/wp#"))
    FILTER(?i a wp:Interaction)
  }}
}}
GROUP BY ?type
ORDER BY DESC(?n)
"""), "interaction_types.csv")
int_types

In [ ]:
print("Querying per-species metrics...")
per_species = save_csv(run_query(f"""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi: <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?species
       (COUNT(DISTINCT ?pathway)  AS ?pathways)
       (COUNT(DISTINCT ?gene)     AS ?genes)
       (COUNT(DISTINCT ?protein)  AS ?enzymes)
       (COUNT(DISTINCT ?met)      AS ?metabolites)
WHERE {{
  GRAPH <{G_TAXONOMY}> {{
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
  GRAPH <{G_CORE}> {{
    ?node wp:isPartOf ?pathway .
    OPTIONAL {{ ?gene a wp:GeneProduct ; wp:isPartOf ?pathway . }}
    OPTIONAL {{ ?protein a wp:Protein  ; wp:isPartOf ?pathway . }}
    OPTIONAL {{ ?met a wp:Metabolite   ; wp:isPartOf ?pathway . }}
  }}
}}
GROUP BY ?species
ORDER BY DESC(?pathways)
"""), "per_species_nrs.csv")
per_species.head(5)

In [ ]:
print("Querying pathway titles...")
titles = save_csv(run_query(f"""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?pwID ?title
WHERE {{
  GRAPH <{G_CORE}> {{
    ?pwID a wp:Pathway ; rdfs:label ?title .
  }}
}}
"""), "pathway_titles.csv")
print(f"  {len(titles):,} pathways")

---
## 2. Normalize column names

In [ ]:
def norm(df, count_name):
    """Rename count column and standardise pwID/title."""
    df = df.copy()
    cc = find_col(df, "count")
    df = df.rename(columns={cc: count_name})
    df[count_name] = pd.to_numeric(df[count_name], errors="coerce").fillna(0)
    try:
        df = df.rename(columns={find_col(df, "pwid"): "pwID"})
        df["pwID"] = df["pwID"].astype(str)
    except ValueError:
        pass
    try:
        df = df.rename(columns={find_col(df, "title"): "title"})
    except ValueError:
        pass
    return df

genes       = norm(genes,       "genes")
metabolites = norm(metabolites, "metabolites")
enzymes     = norm(enzymes,     "enzymes")
conversions = norm(conversions, "conversions")
species_pw  = norm(species_pw,  "species")

# Interaction types
tc = find_col(int_types, "type")
nc = find_col(int_types, "n") if any("n" == c.lower() for c in int_types.columns) else find_col(int_types, "count")
int_types = int_types.rename(columns={tc: "type", nc: "n"})
int_types["n"] = pd.to_numeric(int_types["n"], errors="coerce").fillna(0).astype(int)

WP_INTERACTION = "http://vocabularies.wikipathways.org/wp#Interaction"
total_interactions = int(int_types.loc[int_types["type"] == WP_INTERACTION, "n"].iloc[0])

# per_species
per_species.columns = [c.strip().strip('"') for c in per_species.columns]
for c in ["pathways","genes","enzymes","metabolites"]:
    if c in per_species.columns:
        per_species[c] = pd.to_numeric(per_species[c], errors="coerce").fillna(0)
per_species = per_species.rename(columns={find_col(per_species, "species"): "species"})
per_species["species"] = per_species["species"].apply(
    lambda x: x.split("/")[-1].replace("NCBITaxon_","ncbi:") if str(x).startswith("http") else x
)

# titles
titles = titles.rename(columns={find_col(titles,"pwid"): "pwID", find_col(titles,"title"): "title"})

print("✅ Data ready")
print(f"  Pathways with gene data:       {len(genes):,}")
print(f"  Pathways with metabolite data: {len(metabolites):,}")
print(f"  Total interactions:            {total_interactions:,}")
print(f"  Species in per-species table:  {len(per_species):,}")

---
## 3. Figure: Overview bar chart (log scale)

In [ ]:
overview = {
    "Pathways":     len(genes),
    "Genes":        int(genes["genes"].sum()),
    "Metabolites":  int(metabolites["metabolites"].sum()),
    "Enzymes":      int(enzymes["enzymes"].sum()),
    "Interactions": total_interactions,
}

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(list(overview.keys()), list(overview.values()))
ax.set_yscale("log")
ax.set_ylabel("Count (log scale)")
ax.set_title("PlantMetWiki — content overview")
for bar, val in zip(bars, overview.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.1,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_overview_barlog")
plt.show()

---
## 4. Figure: Cumulative coverage curves

In [ ]:
def cumulative_curve(df, col):
    s = df[col].sort_values(ascending=False)
    total = s.sum()
    return (s.cumsum() / total).values if total > 0 else s.cumsum().values

curves = {
    "Genes":           cumulative_curve(genes,       "genes"),
    "Enzymes":         cumulative_curve(enzymes,     "enzymes"),
    "Metabolites":     cumulative_curve(metabolites, "metabolites"),
    "Conversions":     cumulative_curve(conversions, "conversions"),
    "Species/pathway": cumulative_curve(species_pw,  "species"),
}
styles = {
    "Genes":           dict(linestyle="-",  marker="o", markevery=100),
    "Enzymes":         dict(linestyle="--", marker="s", markevery=100),
    "Metabolites":     dict(linestyle="-.", marker="^", markevery=100),
    "Conversions":     dict(linestyle=":",  marker="x", markevery=100),
    "Species/pathway": dict(linestyle="--", marker="D", markevery=100),
}
fig, ax = plt.subplots(figsize=(7, 4.8))
for label, curve in curves.items():
    ax.plot(range(1, len(curve)+1), curve, label=label, linewidth=2, **styles[label])
ax.set_xlabel("Pathways (ranked by contribution)")
ax.set_ylabel("Cumulative fraction of total")
ax.set_ylim(0, 1.01)
ax.set_title("Cumulative coverage of PlantMetWiki content")
ax.legend()
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_cumulative_coverage")
plt.show()

---
## 5. Figure: Interaction types (2-panel)

In [ ]:
label_map = {
    "http://vocabularies.wikipathways.org/wp#DirectedInteraction":   "Directed interaction",
    "http://vocabularies.wikipathways.org/wp#Conversion":            "Biochemical conversion",
    "http://vocabularies.wikipathways.org/wp#Catalysis":             "Catalysis",
    "http://vocabularies.wikipathways.org/wp#TranscriptionTranslation": "Transcription/translation",
    "http://vocabularies.wikipathways.org/wp#Inhibition":            "Inhibition",
    "http://vocabularies.wikipathways.org/wp#Stimulation":           "Stimulation",
    "http://vocabularies.wikipathways.org/wp#Binding":               "Binding",
}
sub = int_types[int_types["type"] != WP_INTERACTION].copy()
sub["label"] = sub["type"].map(label_map).fillna(sub["type"])
sub["pct"]   = 100 * sub["n"] / total_interactions
sub = sub.sort_values("n", ascending=True)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 4.2),
                                gridspec_kw={"width_ratios": [1, 3]})
ax0.bar(["Total\ninteractions"], [total_interactions])
ax0.set_ylabel("Count"); ax0.set_title("A")
ax0.text(0, total_interactions, f"{total_interactions:,}",
         ha="center", va="bottom", fontsize=10)
ax0.spines[["top","right"]].set_visible(False)

ax1.barh(sub["label"], sub["n"])
ax1.set_xlabel("Number of interactions"); ax1.set_title("B")
xmax = sub["n"].max()
for y, (n, pct) in enumerate(zip(sub["n"], sub["pct"])):
    ax1.text(n + xmax*0.01, y, f"{n:,}  ({pct:.1f}%)", va="center", fontsize=9)
ax1.set_xlim(0, xmax*1.25)
ax1.spines[["top","right"]].set_visible(False)

fig.suptitle("Interaction types in PlantMetWiki", y=1.02)
plt.tight_layout()
save_fig(fig, "plantmetwiki_interaction_types")
plt.show()

---
## 6. Figure: Species metrics — stacked bar (top 50)

In [ ]:
top50 = per_species.sort_values("pathways", ascending=False).head(50).copy()
stack_metrics = [c for c in ["genes","enzymes","metabolites"] if c in top50.columns]
COLORS = {"genes": "C1", "enzymes": "C2", "metabolites": "C3"}

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(top50))
bottom = np.zeros(len(top50))
for m in stack_metrics:
    ax.bar(x, top50[m].values, bottom=bottom, label=m.capitalize(), color=COLORS[m])
    bottom += top50[m].values
ax.set_xticks(x)
ax.set_xticklabels(top50["species"], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Count")
ax.set_title("PlantMetWiki content by species (top 50)")
ax.legend(ncol=3, frameon=False)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_species_metrics_stacked_top50")
plt.show()

---
## 7. Figure: Scatter — genes vs metabolites per pathway (sized by species count)

In [ ]:
merged = metabolites[["pwID","metabolites"]].merge(
    genes[["pwID","genes"]], on="pwID", how="outer"
).merge(
    species_pw[["pwID","species"]], on="pwID", how="outer"
).fillna(0)
merged = merged.merge(titles[["pwID","title"]], on="pwID", how="left")
merged = merged[(merged["metabolites"] > 0) | (merged["genes"] > 0)].copy()

sp_min, sp_max = merged["species"].min(), merged["species"].max()
size = 6 + (merged["species"] - sp_min) / max(sp_max - sp_min, 1) * 154

bins   = sorted(set([0, 1, 2, 5, 10, int(sp_max)]))
blabels = [f"{bins[i-1]+1 if i>1 else 0}-{bins[i]}" if bins[i-1]+1 != bins[i]
           else str(bins[i]) for i in range(1, len(bins))]
merged["sp_bin"]  = pd.cut(merged["species"], bins=bins, include_lowest=True,
                            right=True, labels=blabels)
merged["sp_code"] = merged["sp_bin"].cat.codes

fig, ax = plt.subplots(figsize=(6.8, 5.4))
ax.scatter(merged["metabolites"], merged["genes"],
           s=size, c=merged["sp_code"], cmap="viridis", alpha=0.35, edgecolors="none")
ax.set_xlabel("Metabolites per pathway")
ax.set_ylabel("Genes per pathway")
ax.set_title("Pathway content in PlantMetWiki")

cmap = plt.get_cmap("viridis")
norm = mpl.colors.Normalize(vmin=0, vmax=max(1, len(blabels)-1))
handles = [Line2D([0],[0], marker="o", linestyle="None",
                  markerfacecolor=cmap(norm(i)), markeredgecolor="none",
                  markersize=8, alpha=0.8) for i in range(len(blabels))]
ax.legend(handles, blabels, title="Species count", loc="best", frameon=False)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "scatter_genes_vs_metabolites_size_species")

merged.sort_values(["genes","metabolites"], ascending=False)[
    ["pwID","title","genes","metabolites","species"]
].head(50).to_csv(OUT_DIR / "top_pathways_by_genes.csv", index=False)
print("Saved top_pathways_by_genes.csv")
plt.show()

---
## Sandbox — add new queries and figures here

In [ ]:
# New query or figure
pass